<a href="https://colab.research.google.com/github/zogydinesh/ITA-0616/blob/main/Program3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Create the standard PlayTennis dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)
df.to_csv('playtennis.csv', index=False)
print("playtennis.csv generated successfully!")

In [4]:
import pandas as pd
import math

# ---- 1. Generate & Save CSV (Ensures file is present) ----
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}
pd.DataFrame(data).to_csv('playtennis.csv', index=False)

# ---- 2. Load CSV ----
df = pd.read_csv('playtennis.csv')
target = 'PlayTennis'
features = [c for c in df.columns if c != target]

# ---- 3. Entropy ----
def entropy(data):
    total = len(data)
    return -sum((c / total) * math.log2(c / total) for c in data[target].value_counts())

# ---- 4. Information Gain ----
def info_gain(data, attr):
    total = len(data)
    remainder = 0
    for v in data[attr].unique():
        subset = data[data[attr] == v]
        remainder += (len(subset) / total) * entropy(subset)
    return entropy(data) - remainder

# ---- 5. ID3 algorithm ----
def id3(data, features):
    labels = data[target]
    if labels.nunique() == 1:              # pure node -> leaf
        return labels.iloc[0]
    if not features:
        return labels.mode()[0]
    gains = {f: info_gain(data, f) for f in features}
    best = max(gains, key=gains.get)       # highest gain
    tree = {best: {}}
    for v in data[best].unique():
        subset = data[data[best] == v]
        rest = [f for f in features if f != best]
        tree[best][v] = id3(subset, rest)
    return tree

# ---- 6. Print the tree with visual structure ----
def print_tree(tree, indent="", is_last=True):
    if not isinstance(tree, dict):
        print(indent + "└── " + str(tree))
        return

    attr = next(iter(tree))
    print(indent + ("└── " if is_last else "├── ") + f"[{attr}]")

    sub_indent = indent + ("    " if is_last else "│   ")
    branches = list(tree[attr].items())

    for i, (val, sub) in enumerate(branches):
        branch_last = (i == len(branches) - 1)
        print(sub_indent + ("└── " if branch_last else "├── ") + f"{val}")

        next_indent = sub_indent + ("    " if branch_last else "│   ")
        if isinstance(sub, dict):
            print_tree(sub, next_indent, is_last=True)
        else:
            print(next_indent + "└── " + f"Result: {sub}")

# ---- 7. Classify a new sample ----
def classify(tree, sample):
    if not isinstance(tree, dict):
        return tree
    attr = next(iter(tree))
    return classify(tree[attr][sample[attr]], sample)

# ---- Run ----
print(f"Entropy of dataset: {entropy(df):.4f}\n")
print("Information Gain at root:")
for f in features:
    print(f"  {f}: {info_gain(df, f):.4f}")

tree = id3(df, features)
print("\nDecision Tree Structure:")
print_tree(tree)

new_sample = {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
print("\nNew sample:\n", new_sample)
print("Predicted class (PlayTennis):", classify(tree, new_sample))

Entropy of dataset: 0.9403

Information Gain at root:
  Outlook: 0.2467
  Temperature: 0.0292
  Humidity: 0.1518
  Wind: 0.0481

Decision Tree Structure:
└── [Outlook]
    ├── Sunny
    │   └── [Humidity]
    │       ├── High
    │       │   └── Result: No
    │       └── Normal
    │           └── Result: Yes
    ├── Overcast
    │   └── Result: Yes
    └── Rain
        └── [Wind]
            ├── Weak
            │   └── Result: Yes
            └── Strong
                └── Result: No

New sample:
 {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted class (PlayTennis): No
